# 02 Database, Apps and Support Model

Oracle's software franchise funds the AI buildout and keeps the downside
from being a pure infrastructure balance-sheet story. This notebook
separates cloud applications, database support and services/hardware.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    DcfInputs,
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    discounted_cash_flow,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/oracle_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/oracle_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
cases = ["bear", "base", "bull"]

def segment_case(segment):
    return (
        assumptions[assumptions["segment"].eq(segment)]
        .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
        .reindex(cases)
    )

software_segments = {
    segment: segment_case(segment)
    for segment in ["Cloud Apps", "Database Support", "Services Hardware"]
}
software_segments["Cloud Apps"]

In [ ]:
segment_outputs = []
for segment, table in software_segments.items():
    output = table.assign(
        ebitda_usd_bn=lambda df: df["FY2030 revenue"] * df["EBITDA margin"],
        present_value_usd_bn=lambda df: df["FY2030 revenue"]
        * df["EBITDA margin"]
        * df["EV/EBITDA multiple"]
        * df["discount factor"],
    )
    output["segment"] = segment
    segment_outputs.append(output.reset_index())

software_value = pd.concat(segment_outputs, ignore_index=True)
software_value.pivot_table(
    index=["segment", "case"],
    values=["FY2030 revenue", "ebitda_usd_bn", "present_value_usd_bn"],
)

In [ ]:
apps_margin_multiple_grid = build_sensitivity_grid(
    row_values=[35, 40, 45, 50, 55],
    column_values=[13, 16, 19, 22],
    formula=lambda revenue, multiple: revenue * 0.38 * multiple * 0.68,
    row_name="FY2030 cloud apps revenue (USD bn)",
    column_name="EV/EBITDA multiple",
)
apps_margin_multiple_grid

In [ ]:
database_decay_grid = build_sensitivity_grid(
    row_values=[28, 31, 34, 37, 40],
    column_values=[8, 10, 12, 14],
    formula=lambda revenue, multiple: revenue * 0.48 * multiple * 0.68,
    row_name="FY2030 database support revenue (USD bn)",
    column_name="EV/EBITDA multiple",
)
database_decay_grid